## Constant Alpha Monte Carlo Control example of Gymnasium Blackjack

In [ ]:
USE {
    repositories {
        mavenCentral()
        maven("https://central.sonatype.com/repository/maven-snapshots/")
    }
    dependencies {
        implementation("io.github.kotlinrl:integration:0.1.0-SNAPSHOT")
    }
}
%use dataframe
%use kandy

In [ ]:
import io.github.kotlinrl.core.*
import io.github.kotlinrl.integration.gymnasium.*
import io.github.kotlinrl.integration.gymnasium.GymnasiumEnvs.*
import java.io.*

Let's define our hyper-parameters to control training and learning

In [ ]:
val maxStepsPerEpisode = 10
val trainingEpisodes = 1_000_000
val testEpisodes = 300_000
val initialEpsilon = 1.0
val epsilonDecayRate = 0.9999
val minEpsilon = 0.05
val gamma = 0.99
val alpha = 0.05
val fileName = "BlackjackConstantAlphaMonteCarloControl.csv"

Creating the following
- Env (BlackjackEnv = ```Env<List<Any>, Int, Tuple, Discrete>``` based on the Gymnasium
structure)
    - We use a ```TransformState``` wrapper to change the state from ```Tuple``` space - which has observations of type ```List<Any>``` - to a ```MultiDiscrete``` space - which has observations in ```IntArray```
    - The ```MultiDiscrete``` space works perfectly for tabular data like the ```QTable```
- ```EpisodeCallback``` to log starting episodes
- ```StateActionListProvider``` to define the list of Actions for any State.  Blackjack only allows
    - Actions 1=Hit and 0=Stick
    - State is now ```IntArray``` based on the ```MultiDiscreate``` state space
- The ```QTable``` used to capture training information
    - Monte Carlo Control must wait until the end of each Episode to update the ```QTable```

In [ ]:
val env = TransformState(
    env = gymnasium.make<BlackjackEnv>(Blackjack_v1, render = true),
    transform = { state -> state.map { it as Int }.toIntArray() },
    observationSpace = MultiDiscrete(intArrayOf(32, 11, 2))
)
val episodeLogger = object : EpisodeCallback<IntArray, Int> {
    override fun onEpisodeStart(episode: Int) {
        if (episode > 0 && episode % 100_000 == 0) println("Starting episode $episode")
    }
}
val actionListProvider = StateActionListProvider<IntArray, Int> { listOf(0, 1) }

Next we create the training Agent using the Monte Carlo Control
- We use an Epsilon Greedy Policy with a decaying epsilon to encourage convergence (experimenting with a constant epsilon would lead to different results) for the exploration factor
- The Epsilon Greedy Policy randomly chooses a number.
    - If it is less than the epsilon value it uses a Random Policy selection
    - Otherwise it uses a Greedy Policy to select the best action from the ```QTable```

The Trainer uses the env and agent with a max steps per episode and trains for the number of training episodes
- We register the ```ConstantAlphaMonteCarloControl``` with a ```gamma``` value and and ```alpha``` constant as a ```EpisodeCallback``` so that when the episode completes, it updates the ```QTable```
- We also register the episode logger
- We then train for the number of training episodes
- When training completes we save the ```QTable``` for later use

In [ ]:
val trainingQtable = QTable(32, 11, 2, 2)
if(File(fileName).exists()) {
    trainingQtable.load(fileName)
    println("QTable loaded from file")
}
val trainingAgent = agent(
    id = "training",
    policy = epsilonGreedyPolicy(
        stateActionListProvider = actionListProvider,
        explorationFactor = decayingEpsilon(
            factor = initialEpsilon,
            minFactor = minEpsilon,
            decayRate = epsilonDecayRate
        ),
        qTable = trainingQtable
    )
)
val trainer = episodicTrainer(
    env = env,
    agent = trainingAgent,
    maxStepsPerEpisode = maxStepsPerEpisode,
    callbacks = listOf(
        contantAlphaMonteCarloControl(trainingQtable, gamma, alpha),
        episodeLogger
    )
)
println("Starting training")
val training = trainer.train(trainingEpisodes)
trainingQtable.save(fileName)

Once training is complete, we create the following
- A new ```QTable``` with the same shape, and load the training data
- A new test ```Agent``` using a ```GreedyPolicy``` against the ```QTable``` with loaded weights
- The Greedy Policy always chooses the best action from the ```QTable```
    - It performs the best action given the state: ```(playerSum, dealerSum, usableAce)```

We then test for the number of testing episodes to compare the episode results (i.e. the average reward achieved)

In [ ]:
val testingQtable = QTable(32, 11, 2, 2)
testingQtable.load(fileName)

In [ ]:
val recordEnv = RecordVideo(env = env, folder = "videos/on_policy_mcc", 100_000)
val testingAgent = agent(
    id = "testing",
    policy = greedyPolicy(
        qTable = testingQtable
    )
)
val tester = episodicTrainer(
    env = recordEnv,
    agent = testingAgent,
    maxStepsPerEpisode = maxStepsPerEpisode,
    callbacks = listOf(object : EpisodeCallback<IntArray, Int> {
        override fun onEpisodeStart(episode: Int) {
            if (episode > 0 && episode % 10_000 == 0) println("Starting episode $episode")
        }
    })
)
println("Starting testing")
val test = tester.train(testEpisodes)

Compare the average results.  We also check if the Agent has learned:
> - Q-value for [20, 10, 1, Stick]
> - Q-value for [20, 10, 1, Hit]:

This is a classic scenario:
> - Player has 20
> - Dealer shows 10
> - Usable ace is true
> - The Agent has learned that:
> - Sticking has a positive expected return (≈ 0.45)
> - Hitting has poor returns (possibly busted → 0.0)
> - ✅ That aligns with good Blackjack strategy: 20 is a strong hand — you usually stand.

In [ ]:
println("Training Results: ${training.episodeRewards.sum() / training.episodeRewards.size}")
println("Test Results: ${test.episodeRewards.sum() / test.episodeRewards.size}")
println("Q-value for [20, 10, 1, Stick]: ${testingQtable[intArrayOf(20, 10, 1), 0]}")
println("Q-value for [20, 10, 1, Hit]:   ${testingQtable[intArrayOf(20, 10, 1), 1]}")

### Here is basic strategy for playing Blackjack

In [ ]:
val dfBasic = buildList {
    for (playerSum in 12..21) {
        for (dealerCard in 1..10) {
            for (usableAce in listOf(0, 1)) {
                val action = when {
                    // Soft totals (usable ace == 1)
                    usableAce == 1 -> when (playerSum) {
                        in 19..21 -> 0 // Always Stick
                        18 -> if (dealerCard in 9..10 || dealerCard == 1) 1 else 0 // Stick unless dealer is strong
                        else -> 1 // Hit
                    }

                    // Hard totals (no usable ace)
                    else -> when (playerSum) {
                        in 17..21 -> 0 // Always Stick
                        in 13..16 -> if (dealerCard in 2..6) 0 else 1 // Stick if dealer shows weak card
                        12 -> if (dealerCard in 4..6) 0 else 1 // Stick if dealer shows 4-6
                        else -> 1 // Hit
                    }
                }

                val actionLabel = if (action == 0) "Stick" else "Hit"
                val label = if (action == 0) "S" else "H"

                add(
                    mapOf(
                        "playerSum" to playerSum,
                        "dealerCard" to dealerCard,
                        "usableAce" to if (usableAce == 1) "Usable Ace" else "No Ace",
                        "action" to actionLabel,
                        "label" to label
                    )
                )
            }
        }
    }
}.flatMap { it.entries }.groupBy({ it.key }, { it.value }).toDataFrame()
plotGrid(dfBasic.groupBy("usableAce").map { (aceLabel, group) ->
    group.plot {
        layout.title = "Basic Strategy: Hit (H) vs Stick (S) – ${aceLabel[0]}"

        heatmap("playerSum", "dealerCard") {
            fillColor(group["action"]) {
                scale = categorical(
                    "Hit" to Color.GREEN,
                    "Stick" to Color.RED
                )
            }
            borderLine {
                width = 0.5
                color = Color.BLACK
            }
        }

        text {
            x("playerSum")
            y("dealerCard")
            label(group["label"])
        }

        x.axis.name = "Player Sum"
        y.axis.name = "Dealer Up Card"
    }
})

Let's plot our results

In [ ]:
val dfAgent = buildList {
    for (playerSum in 12..21) {
        for (dealerCard in 1..10) {
            for (usableAce in listOf(0, 1)) {
                val state = intArrayOf(playerSum, dealerCard, usableAce)
                val action = testingQtable.bestAction(state)
                val actionLabel = if (action == 1) "Hit" else "Stick"
                val label = if (action == 1) "H" else "S"

                add(
                    mapOf(
                        "playerSum" to playerSum,
                        "dealerCard" to dealerCard,
                        "usableAce" to if (usableAce == 1) "Usable Ace" else "No Ace",
                        "action" to actionLabel,
                        "label" to label
                    )
                )
            }
        }
    }
}.flatMap { it.entries }.groupBy({ it.key }, { it.value }).toDataFrame()
plotGrid(dfAgent.groupBy("usableAce").map { (aceLabel, group) ->
    group.plot {
        layout.title = "Agent Policy: Hit (H) vs Stick (S) – ${aceLabel[0]}"

        heatmap("playerSum", "dealerCard") {
            fillColor(group["action"]) {
                scale = categorical(
                    "Hit" to Color.GREEN,
                    "Stick" to Color.RED
                )
            }
            borderLine {
                width = 0.5
                color = Color.BLACK
            }
        }

        text {
            x("playerSum")
            y("dealerCard")
            label(group["label"])
        }

        x.axis.name = "Player Sum"
        y.axis.name = "Dealer Up Card"
    }
})

In [ ]:
val folder = File(recordEnv.folder)
for(file in folder.listFiles()!!.filter { it.isDirectory }) {
    displayVideo(File(folder, file.name))
}